In [1]:
import json
from collections import defaultdict
    
with open('data/reddit_comment_body_dec_2024.json', 'r') as f:
    data = json.load(f)
    

# Create a dictionary to store messages by author
author_messages = defaultdict(list)
for item in data:
    author_messages[item['author']].append(item['body'])

2 Author Dataset

In [6]:
import random

# Get eligible authors (with at least two messages)
eligible_authors = [auth for auth, msgs in author_messages.items() if len(msgs) >= 2]
if len(eligible_authors) < 2:
    raise ValueError("Not enough authors with at least 2 messages.")

# Randomly choose 2 authors
author1, author2 = random.sample(eligible_authors, 2)

def create_positive_pairs(msgs, n_pairs):
    msgs_copy = msgs[:]  # copy to avoid modifying original list
    random.shuffle(msgs_copy)
    pairs = []
    for i in range(n_pairs):
        pairs.append((msgs_copy[2 * i], msgs_copy[2 * i + 1]))
    return pairs

# Determine how many pairs we can form from each author so they're balanced.
n_pairs_author1 = len(author_messages[author1]) // 2
n_pairs_author2 = len(author_messages[author2]) // 2
n_pairs = min(n_pairs_author1, n_pairs_author2)
if n_pairs == 0:
    raise ValueError("Not enough messages to form pairs for both authors.")

# Create positive examples (same-author pairs with label 1)
pos_pairs_auth1 = create_positive_pairs(author_messages[author1], n_pairs)
pos_pairs_auth2 = create_positive_pairs(author_messages[author2], n_pairs)

training_examples = []
for pair in pos_pairs_auth1:
    training_examples.append((pair[0], pair[1], 1, author1, author1))
for pair in pos_pairs_auth2:
    training_examples.append((pair[0], pair[1], 1, author2, author2))

# For negative examples (different-author pairs with label 0), 
# use the messages selected in the positive pairing to balance the examples.
auth1_msgs = [msg for pair in pos_pairs_auth1 for msg in pair]
auth2_msgs = [msg for pair in pos_pairs_auth2 for msg in pair]

random.shuffle(auth1_msgs)
random.shuffle(auth2_msgs)

n_negative = min(len(auth1_msgs), len(auth2_msgs))
for i in range(n_negative):
    training_examples.append((auth1_msgs[i], auth2_msgs[i], 0, author1, author2))

# Shuffle training examples
random.shuffle(training_examples)

# Check balance and sample output
pos_count = sum(1 for ex in training_examples if ex[2] == 1)
neg_count = sum(1 for ex in training_examples if ex[2] == 0)
print("Selected authors:", author1, author2)
print("Number of positive examples:", pos_count)
print("Number of negative examples:", neg_count)
print("Sample training examples:")
for ex in training_examples[:5]:
    print(ex)

Selected authors: Ok-Meeting-7154 Then_Calligrapher855
Number of positive examples: 3918
Number of negative examples: 3918
Sample training examples:
('You’re right', 'Need that booty. Coffee. I mean need that coffee', 0, 'Ok-Meeting-7154', 'Then_Calligrapher855')
('When they treat people purely specifically someone who’s just doing their job.', '#WTF DID I JUST READ', 1, 'Then_Calligrapher855', 'Then_Calligrapher855')
('You’re welcome. Lol', 'So hard. Harder than life', 1, 'Then_Calligrapher855', 'Then_Calligrapher855')
('Hard to tell when you’re in the middle of it', 'Ok, you’re cut off.', 1, 'Ok-Meeting-7154', 'Ok-Meeting-7154')
('Orange (as in fake tan)', 'Great minds', 1, 'Ok-Meeting-7154', 'Ok-Meeting-7154')


In [7]:
import torch
from transformers import BertModel

import torch.nn as nn

class SiameseBERT(nn.Module):
    def __init__(self, pretrained_model_name="bert-base-uncased", hidden_size=768, dropout_prob=0.1):
        super(SiameseBERT, self).__init__()
        self.bert = BertModel.from_pretrained(pretrained_model_name)
        self.dropout = nn.Dropout(dropout_prob)
        # Optionally, a classifier head based on the difference of embeddings.
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )
        
    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2):
        out1 = self.bert(input_ids=input_ids1, attention_mask=attention_mask1)
        embed1 = out1.pooler_output  # [batch_size, hidden_size]
        embed1 = self.dropout(embed1)
        
        out2 = self.bert(input_ids=input_ids2, attention_mask=attention_mask2)
        embed2 = out2.pooler_output  # [batch_size, hidden_size]
        embed2 = self.dropout(embed2)
        
        # For classification: use the absolute difference of the embeddings.
        diff = torch.abs(embed1 - embed2)
        prob = self.classifier(diff)  # probability that both texts were written by same author
        return prob, embed1, embed2

def contrastive_loss(embedding1, embedding2, label, margin=1.0):
    """
    Computes the contrastive loss.
    label: 1 if same author, 0 otherwise.
    """
    # Calculate Euclidean distance between embeddings
    distance = torch.norm(embedding1 - embedding2, p=2, dim=1)
    # Contrastive loss from Hadsell et al.
    loss = label * torch.pow(distance, 2) + (1 - label) * torch.pow(torch.clamp(margin - distance, min=0.0), 2)
    return loss.mean()

# Example usage (ensure the tokenizer is defined in another cell if needed):
# tokenizer = ... (a pretrained BERT tokenizer)
# texts1 = ["sample text 1", "sample text 2"]
# texts2 = ["another text 1", "another text 2"]
# Encode texts:
# encoding1 = tokenizer(texts1, padding=True, truncation=True, return_tensors="pt")
# encoding2 = tokenizer(texts2, padding=True, truncation=True, return_tensors="pt")
#
# model = SiameseBERT()
# prob, emb1, emb2 = model(encoding1['input_ids'], encoding1['attention_mask'],
#                            encoding2['input_ids'], encoding2['attention_mask'])
# labels = torch.tensor([1, 0], dtype=torch.float)  # example labels (1: same author, 0: different)
#
# loss = contrastive_loss(emb1, emb2, labels)
# print("Contrastive loss:", loss)

In [8]:
from transformers import BertTokenizer
import torch

# Initialize tokenizer (if not already defined)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Extract texts and labels from training_examples (already defined in a previous cell)
texts1 = [ex[0] for ex in training_examples]
texts2 = [ex[1] for ex in training_examples]
labels = [ex[2] for ex in training_examples]

# Tokenize each set of texts
encoded_inputs1 = tokenizer(texts1, padding=True, truncation=True, return_tensors="pt")
encoded_inputs2 = tokenizer(texts2, padding=True, truncation=True, return_tensors="pt")

# Convert labels to a tensor
labels = torch.tensor(labels, dtype=torch.float)

# Pack tokenized data into a dictionary
training_data = {
    "input_ids1": encoded_inputs1["input_ids"],
    "attention_mask1": encoded_inputs1["attention_mask"],
    "input_ids2": encoded_inputs2["input_ids"],
    "attention_mask2": encoded_inputs2["attention_mask"],
    "labels": labels
}

print("Tokenization complete. Example input_ids1 for first example:")
print(training_data["input_ids1"][0])

Tokenization complete. Example input_ids1 for first example:
tensor([ 101, 2017, 1521, 2128, 2157,  102,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import TensorDataset, DataLoader

import torch.nn.functional as F

from tqdm.notebook import tqdm

# Prepare dataset using tensors from training_data defined earlier
dataset = TensorDataset(
    training_data["input_ids1"],
    training_data["attention_mask1"],
    training_data["input_ids2"],
    training_data["attention_mask2"],
    training_data["labels"]
)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate model and move to device
model = SiameseBERT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
n_epochs = 3
batch_size = 8

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 0
for train_index, val_index in kf.split(dataset):
    fold += 1
    print(f"\nFold {fold}")
    
    train_subset = torch.utils.data.Subset(dataset, train_index)
    val_subset = torch.utils.data.Subset(dataset, val_index)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size)
    
    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        # Use tqdm progress bar on the training loop
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            input_ids1, mask1, input_ids2, mask2, labels = batch
            input_ids1 = input_ids1.to(device)
            mask1 = mask1.to(device)
            input_ids2 = input_ids2.to(device)
            mask2 = mask2.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            prob, emb1, emb2 = model(input_ids1, mask1, input_ids2, mask2)
            print(f"first 50 probabilities: {prob.squeeze()[:50]}")
            loss = contrastive_loss(emb1, emb2, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * labels.size(0)
            # Compute accuracy (threshold probability at 0.5)
            preds = (prob.squeeze() > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        avg_loss = train_loss / total
        train_acc = correct / total * 100
        print(f"Epoch {epoch} - Loss: {avg_loss:.4f} - Training Accuracy: {train_acc:.2f}%")
        
    # Validation after training fold
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids1, mask1, input_ids2, mask2, labels = batch
            input_ids1 = input_ids1.to(device)
            mask1 = mask1.to(device)
            input_ids2 = input_ids2.to(device)
            mask2 = mask2.to(device)
            labels = labels.to(device)
            
            prob, _, _ = model(input_ids1, mask1, input_ids2, mask2)
            preds = (prob.squeeze() > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total * 100
    print(f"Fold {fold} Validation Accuracy: {val_acc:.2f}%")



Fold 1


Epoch 1:   0%|          | 0/784 [00:00<?, ?it/s]

first 50 probabilities: tensor([0.4547, 0.4930, 0.4567, 0.5521, 0.4907, 0.5404, 0.4974, 0.5570],
       grad_fn=<SliceBackward0>)
